(sec:computational_optimisation)=

# Computational Optimisation

Many scientific problems can be formulated as the search for a minimum of an objective function. Computational optimisation has very broad applications in chemistry: fitting parameters, optimising molecular structures, and performing variational calculations, to name just a few. In this section, we introduce gradient descent as a simple iterative optimisation method and apply it to optimise the yield of a chemical reaction as a function of temperature and reaction time.
We then turn to `scipy.optimize.minimize`, which provides library implementations for optimisation problems of this kind as well as access to more sophisticated optimisation algorithms.

## Theoretical Foundations

We consider a function $f(\vec{x})$ of several variables and want to find a
parameter vector $\vec{x}^*$ for which $f$ becomes minimal.


### Gradient Descent

Gradient descent updates the current parameter vector in the direction of
steepest descent,
```{math}
:enumerated: true
:label: eq:compopt-gradient-descent
\vec{x}^{k+1} = \vec{x}^k - \alpha \nabla f(\vec{x}^k) \, .
```
Here, $\alpha$ is the step size and $\nabla f(\vec{x}^k)$ is the gradient at the
current iterate. The iteration is repeated until a stopping criterion is met.

:::{note} Why the negative gradient?
:class: dropdown

The gradient points in the direction of steepest increase of the function.
Therefore, the negative gradient points towards a local decrease. This does
not guarantee convergence to the global minimum, but it often provides a good
local search direction.
:::

:::{note} Step size and convergence
:class: dropdown

The step size $\alpha$ must be chosen with care. If it is too small, the
iteration progresses very slowly. If it is too large, the updates may jump
across the minimum and the method can become unstable.
:::


### Finite Difference in Several Dimensions

To apply Eq. {eq}`eq:compopt-gradient-descent`, we need the gradient. In
{ref}`sec:root_finding`, we introduced the one-dimensional central finite-
difference formula. For a function of several variables, the same idea is
applied component by component:
```{math}
:enumerated: true
:label: eq:compopt-finite-difference
\frac{\partial f}{\partial x_i}(\vec{x})
\approx
\frac{f(\vec{x} + h\hat{e}_i) - f(\vec{x} - h\hat{e}_i)}{2h} \, .
```
Here, $\hat{e}_i$ is the unit vector in direction $i$. Repeating this for all
components yields an approximation to the full gradient vector.

:::{note} Why unit vectors appear here
:class: dropdown

The unit vector $\hat{e}_i$ changes only one component of $\vec{x}$ at a time.
This lets us isolate the partial derivative with respect to $x_i$ while
keeping all other components fixed.
:::


## Implementation


Since we will use Numpy arrays to represent the parameter vector $\vec{x}$, we at first import Numpy.

In [1]:
import numpy as np

### Finite Difference Gradient

The function below implements Eq. {eq}`eq:compopt-finite-difference`. It
expects a scalar objective function `func`, a parameter vector `x_vec`, and a
finite-difference step size `h`. The return value is a NumPy array containing
all components of the approximate gradient.


In [2]:
def finite_difference_gradient(func, x_vec, h=1e-6):
    ndim = len(x_vec)
    grad = np.zeros(ndim)

    for i in range(ndim):
        e_i = np.zeros(ndim)
        e_i[i] = 1.0
        grad[i] = (
            func(x_vec + h * e_i) - func(x_vec - h * e_i)
        ) / (2 * h)

    return grad

The implementation mirrors the equation almost directly. First, we determine
how many components the parameter vector has and create an array `grad` of the
same length. The `for` loop then visits all components one after another. In
each step, we build the corresponding unit vector `e_i`, evaluate the
objective function at the shifted points `x_vec + h * e_i` and `x_vec - h * e_i`,
and store the resulting finite-difference quotient in the gradient array.

This is the multidimensional analogue of the one-dimensional finite
difference from {ref}`sec:root_finding`. The only new idea is that the same
formula must be applied separately to each component.


### Gradient Descent

The next function implements Eq. {eq}`eq:compopt-gradient-descent`. It takes
an objective function `func`, an initial parameter vector `x0`, a step size
`alpha`, a gradient tolerance `gtol`, and a maximum number of iterations
`maxiter`. It returns the final parameter vector together with the number of
iterations used.


In [ ]:
def gradient_descent(func, x0, alpha=0.01, gtol=1e-6, maxiter=10000):
    x_vec = np.copy(x0)

    for niter in range(1, maxiter + 1):
        grad = finite_difference_gradient(func, x_vec)

        if np.linalg.norm(grad) < gtol:
            break

        x_vec = x_vec - alpha * grad

    return x_vec, niter

This implementation follows the same iterative structure as our earlier
root-finding algorithms. We start from an initial guess `x0` and create a
copy called `x_vec`, so that the original array is not modified outside the
function. In every iteration, we first evaluate the approximate gradient at
the current point.
After that, the gradient norm is compared with the tolerance `gtol`. If the
norm is already small enough, the convergence criterion is satisfied and the
`break` command leaves the loop immediately. This means that no further update
step is taken once the current iterate is already close enough to a
stationary point. If the criterion is not satisfied, we carry out one
gradient-descent update and continue with the next iteration.

A `for` loop is used here instead of a `while` loop. This is more
appropriate because the number of iterations has a clear upper bound, namely
`maxiter`, even if convergence is not achieved. The loop variable `niter`
counts the iteration steps and is returned at the end.

It is useful to distinguish between a **convergence criterion** and a
**termination criterion**. A convergence criterion specifies when the iterates
are considered sufficiently close to the desired mathematical goal, such as a
solution, optimum, or stationary point. A termination criterion specifies
when the algorithm stops. It may include the convergence criterion, but it
can also include additional conditions such as reaching the maximum number of
iterations, exceeding a time limit, or detecting numerical difficulties.

In this implementation, the convergence criterion is
`np.linalg.norm(grad) < gtol`. The termination criterion is broader: the
algorithm stops either when this convergence criterion is satisfied or when
the maximum number of iterations is reached.

This distinction is important in practice. An algorithm can terminate without
having converged, for example if `maxiter` is too small or if the step size
`alpha` is poorly chosen. Conversely, if the convergence criterion is
satisfied, the algorithm terminates for a mathematically meaningful reason.


## Application

As an example from synthetic chemistry, we consider the epoxidation of
octa-1,7-diene with TBHP in the presence of a PBI·Mo catalyst. The reaction
scheme is shown below.

:::{figure} ../../assets/figures/numerical_foundations/epoxyoctene.svg
:align: center
:width: 100%
Reaction scheme for epoxidation of octa-1,7-diene with TBHP by PBI·Mo catalyst.
:::

We use an simplified version of a published yield model[^1], which expresses 
the yield percentage $Y$ as a function of temperature $T$ and reaction time $t$:
```{math}
:enumerated: true
:label: eq:compopt-yield-model
Y(T, t) = 52.29 + 5.40b + 27.76d + 2.56bd - 3.84b^2 - 20.44d^2
```
with
$$
b = \frac{T - 343}{10}\, , \qquad d = \frac{t - 130}{130}\,,
$$
valid for $T \in [333, 353]$ K and $t \in [0, 260]$ min.

:::{note} Practical Remark
Reaction-condition optimisation is often carried out with gradient-free
optimisation methods, because the yield is measured experimentally or obtained
from a simulation and is usually not available as an analytical function. The
present example is useful for teaching because the paper provides an explicit
surrogate model for the yield.
:::

[^1]: [M. M. R. Bhuiyan, B. Saha, *React. Chem. Eng.* **2004**, *9*, 1036&ndash;1046.](https://pubs.rsc.org/en/content/articlelanding/2024/re/d3re00461a)

We now implement the objective function, which takes a parameter vector `x_vec` 
and returns the negative yield, so that the optimisation problem is a minimisation problem. 
The function first unpacks the temperature and time from the input vector. 
Afterwards, Eq. {eq}`eq:compopt-yield-model` is implemented directly.

In [4]:
def objective_function(x_vec):
    temp, time = x_vec
    b = (temp - 343.0) / 10.0
    d = (time - 130.0) / 130.0

    yield_percentage = (
        52.29
        + 5.40 * b + 27.76 * d + 2.56 * b * d
        - 3.84 * b**2 - 20.44 * d**2
    )

    return -yield_percentage

Just like root finding, the optimisation process starts from an initial guess 
for the parameter vector.
Here we start from the centre point of the model, *i.e.*
$\vec{x}^{\,0} = (343, 130)^{\intercal}$.
Afterwards, we apply the gradient descent method.

In [5]:
x_guess = np.array([343.0, 130.0])

x_opt, niter = gradient_descent(
    objective_function,
    x_guess,
    alpha=1.0,
    gtol=1e-6,
    maxiter=20000,
)

temp_opt, time_opt = x_opt
yield_opt = -objective_function(x_opt)

print('iterations:', niter)
print('opt. temp:', temp_opt, 'K')
print('opt. time:', time_opt, 'min')
print('opt. yield:', yield_opt, '%')

iterations: 5207
opt. temp: 352.4929310137104 K
opt. time: 226.00557477895677 min
opt. yield: 65.10358073763132 %


## Optimisation with SciPy

The same problem can also be solved with
[`scipy.optimize.minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html).
In the following, we use the method `CG`, which stands for *nonlinear
**C**onjugate **G**radient*. 
This is a more sophisticated gradient-based optimisation algorithm that
cleverly chooses its search directions by combining information from 
several previous steps.

In [6]:
from scipy.optimize import minimize

res = minimize(
    objective_function,
    x_guess,
    method='CG',
    jac=lambda x_vec: finite_difference_gradient(objective_function, x_vec),
    options={'gtol': 1e-6, 'maxiter': 20000},
)

temp_opt_scipy, time_opt_scipy = res.x
yield_scipy = -objective_function(res.x)

print('iterations:', res.nit)
print('opt. temp:', temp_opt_scipy, 'K')
print('opt. time:', time_opt_scipy, 'min')
print('opt. yield:', yield_scipy, '%')

iterations: 5
opt. temp: 352.4929452520725 K
opt. time: 226.00610019795621 min
opt. yield: 65.10358073782857 %


The predicted yield agrees with the result of our own implementation, 
but the number of iterations is much smaller. This is the
practical advantage of a more sophisticated optimisation method.

:::{note} Selected Methods in `minimize`
:class: dropdown
The function `minimize` provides access to a range of optimisation methods.
Some common choices are:

- `Nelder-Mead`: gradient-free; uses a simplex that is moved and deformed
  through parameter space. Useful when derivatives are unavailable or noisy,
  but often slower than gradient-based methods.
- `CG`: gradient-based; uses first derivatives and conjugate search
  directions. A good choice for unconstrained smooth problems of moderate
  size.
- `BFGS`: gradient-based quasi-Newton method; builds an approximation to the
  inverse Hessian. Often a strong default choice for smooth unconstrained
  problems.
- `L-BFGS-B`: gradient-based variant of BFGS that is memory-efficient. Useful
  for larger smooth problems.
- `COBYQA`: gradient-free; builds local quadratic models. Useful for smooth
  objective functions when derivatives are not available.
:::

:::{note} Supplying the Gradient
:class: dropdown
The `jac` argument can be used to supply the gradient of the objective
function. If an analytical derivative is available, providing it can make the
optimisation faster and more reliable. If `jac` is not supplied, `minimize`
estimates the gradient numerically by finite differences for methods that use
gradient information. In this example, we pass our own finite-difference
gradient explicitly.
:::

## Self-Study Questions

1. What advantages does `for` loops have over `while` loops in the implementation 
   of iterative algorithms?
2. Why is the norm of the gradient a reasonable quantity to monitor when
   searching for an optimum? Can you think of a situation where this might not be sufficient?
3. In the application example, why does the objective function return the negative yield
   instead of the yield directly?
4. Try changing the initial guess and the step size in the gradient descent implementation. 
   How do these parameters affect convergence? 
   Can you find a combination that leads to divergence?